In [2]:
# Install the packages
! pip3 install --user --no-cache-dir --upgrade "kfp>2" "google-cloud-pipeline-components>2" google-cloud-aiplatform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.6/595.6 kB 27.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


### Restart the Kernel

Once we have installed the additional packages, we need to restart the notebook kernel so it can find the packages.

In [3]:
import os

if not os.getenv("IS_TESTING"):
    # Automatically restart kernel after installs
    import IPython

    app = IPython.Application.instance()
    app.kernel.do_shutdown(True)

### Check the versions of the packages you installed. The KFP SDK version should be >2.


In [1]:
! python3 -c "import kfp; print('KFP SDK version: {}'.format(kfp.__version__))"
! pip3 freeze | grep aiplatform
! python3 -c "import google_cloud_pipeline_components; print('google_cloud_pipeline_components version: {}'.format(google_cloud_pipeline_components.__version__))"

KFP SDK version: 2.7.0
google-cloud-aiplatform==1.70.0
google_cloud_pipeline_components version: 2.17.0


### Import libraries

In [2]:
import kfp
from kfp import dsl
from kfp.dsl import (
    Dataset,
    Input,
    Model,
    Output,
    Artifact,
    ClassificationMetrics,
    Metrics,
    component
)
import google.cloud.aiplatform as aip
from typing import NamedTuple

### Set Project and Pipeline Configurations


In [3]:
# Replace with your Google Cloud project ID and region
PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
PIPELINE_ROOT = "gs://your-temp-bucket"

# Pipeline Components

## 1. Data Ingestion Component

In [4]:
@dsl.component(
    packages_to_install=["pandas", "google-cloud-storage"],
    base_image="python:3.10.7-slim"
)
def download_data_from_gcs(project_id: str, bucket_name: str, file_name: str, dataset: Output[Dataset]):
    """Download the dataset from a Google Cloud Storage bucket."""
    from google.cloud import storage
    import pandas as pd
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    # Initialize the GCS client
    client = storage.Client(project=project_id)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(file_name)

    # Download the file to the component's output path
    blob.download_to_filename(dataset.path + ".csv")
    logging.info(f"Dataset '{file_name}' downloaded from bucket '{bucket_name}'.")

## 2. Data Splitting Component

In [5]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn"],
    base_image="python:3.10.7-slim"
)
def split_data(input_data: Input[Dataset], train_data: Output[Dataset], test_data: Output[Dataset]):
    """Split the data into training and test sets."""
    import pandas as pd
    from sklearn.model_selection import train_test_split
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    # Read the data
    data = pd.read_csv(input_data.path + ".csv", delimiter=';')

    # Split data
    train_df, test_df = train_test_split(data, test_size=0.2, random_state=42)

    # Save train and test data
    train_df.to_csv(train_data.path + ".csv", index=False)
    test_df.to_csv(test_data.path + ".csv", index=False)
    logging.info("Data splitting completed.")

## 3. Data Pre-Processing Component

In [6]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "joblib"],
    base_image="python:3.10.7-slim"
)
def preprocess_data(
    train_data: Input[Dataset],
    test_data: Input[Dataset],
    processed_train_data: Output[Dataset],
    processed_test_data: Output[Dataset],
    scaler_artifact: Output[Artifact]
):
    """Preprocess the data by scaling."""
    import pandas as pd
    from sklearn.preprocessing import StandardScaler
    import joblib
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    # Read the training data
    train_df = pd.read_csv(train_data.path + ".csv")
    X_train = train_df.drop('quality', axis=1)
    y_train = train_df['quality']

    # Fit the scaler on training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
    X_train_scaled_df['quality'] = y_train

    # Save the preprocessed training data
    X_train_scaled_df.to_csv(processed_train_data.path + ".csv", index=False)

    # Save the scaler using scaler_artifact.path directly
    joblib.dump(scaler, scaler_artifact.path)
    logging.info(f"Scaler saved at {scaler_artifact.path}")

    # Read the test data
    test_df = pd.read_csv(test_data.path + ".csv")
    X_test = test_df.drop('quality', axis=1)
    y_test = test_df['quality']

    # Apply the scaler to test data
    X_test_scaled = scaler.transform(X_test)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)
    X_test_scaled_df['quality'] = y_test

    # Save the preprocessed test data
    X_test_scaled_df.to_csv(processed_test_data.path + ".csv", index=False)

    logging.info("Data preprocessing completed.")


## 4. Model Training Components 

We'll create separate components for training each model: KNN, Random Forest, and SVM.

### a. KNN Training Component

In [7]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "imblearn"],
    base_image="python:3.10.7-slim"
)
def train_knn(input_data: Input[Dataset], knn_model: Output[Model]) -> NamedTuple('Outputs', [('accuracy', float)]):
    """Train a KNN classifier with default parameters."""
    import pandas as pd
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import accuracy_score
    from imblearn.over_sampling import RandomOverSampler
    import pickle
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    # Load data
    data = pd.read_csv(input_data.path + ".csv")
    X = data.drop('quality', axis=1)
    y = data['quality']

    # Oversampling to balance classes
    ros = RandomOverSampler(random_state=42)
    X_res, y_res = ros.fit_resample(X, y)

    # Train KNN model with default parameters
    knn = KNeighborsClassifier()
    knn.fit(X_res, y_res)

    y_pred = knn.predict(X_res)
    accuracy = accuracy_score(y_res, y_pred)
    logging.info(f"KNN accuracy: {accuracy}")

    # Save the model
    model_path = knn_model.path + ".pkl"
    with open(model_path, 'wb') as f:
        pickle.dump(knn, f)
    knn_model.metadata["file_type"] = ".pkl"
    knn_model.metadata["model_type"] = "KNN"

    from collections import namedtuple
    Outputs = namedtuple('Outputs', ['accuracy'])
    return Outputs(accuracy=accuracy)

### b. Random Forest Training Component

In [8]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "imblearn"],
    base_image="python:3.10.7-slim"
)
def train_random_forest(input_data: Input[Dataset], rf_model: Output[Model]) -> NamedTuple('Outputs', [('accuracy', float)]):
    """Train a Random Forest classifier with default parameters."""
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score
    from imblearn.over_sampling import RandomOverSampler
    import pickle
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    # Load data
    data = pd.read_csv(input_data.path + ".csv")
    X = data.drop('quality', axis=1)
    y = data['quality']

    # Oversampling to balance classes
    ros = RandomOverSampler(random_state=42)
    X_res, y_res = ros.fit_resample(X, y)

    # Train Random Forest with default parameters
    rf = RandomForestClassifier(class_weight='balanced')
    rf.fit(X_res, y_res)

    y_pred = rf.predict(X_res)
    accuracy = accuracy_score(y_res, y_pred)
    logging.info(f"Random Forest accuracy: {accuracy}")

    # Save the model
    model_path = rf_model.path + ".pkl"
    with open(model_path, 'wb') as f:
        pickle.dump(rf, f)
    rf_model.metadata["file_type"] = ".pkl"
    rf_model.metadata["model_type"] = "RandomForest"

    from collections import namedtuple
    Outputs = namedtuple('Outputs', ['accuracy'])
    return Outputs(accuracy=accuracy)


### c. SVM Training Component

In [9]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "imblearn"],
    base_image="python:3.10.7-slim"
)
def train_svm(input_data: Input[Dataset], svm_model: Output[Model]) -> NamedTuple('Outputs', [('accuracy', float)]):
    """Train an SVM classifier with default parameters."""
    import pandas as pd
    from sklearn.svm import SVC
    from sklearn.metrics import accuracy_score
    from imblearn.over_sampling import RandomOverSampler
    import pickle
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    # Load data
    data = pd.read_csv(input_data.path + ".csv")
    X = data.drop('quality', axis=1)
    y = data['quality']

    # Oversampling to balance classes
    ros = RandomOverSampler(random_state=42)
    X_res, y_res = ros.fit_resample(X, y)

    # Train SVM with default parameters
    svm = SVC(probability=True)
    svm.fit(X_res, y_res)

    y_pred = svm.predict(X_res)
    accuracy = accuracy_score(y_res, y_pred)
    logging.info(f"SVM accuracy: {accuracy}")

    # Save the model
    model_path = svm_model.path + ".pkl"
    with open(model_path, 'wb') as f:
        pickle.dump(svm, f)
    svm_model.metadata["file_type"] = ".pkl"
    svm_model.metadata["model_type"] = "SVM"

    from collections import namedtuple
    Outputs = namedtuple('Outputs', ['accuracy'])
    return Outputs(accuracy=accuracy)


## 5. Model Selection Component

In [10]:
@dsl.component(
    base_image="python:3.10.7-slim"
)
def select_best_model(knn_accuracy: float, rf_accuracy: float, svm_accuracy: float) -> str:
    """Select the best model based on accuracy."""
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    accuracies = {
        'KNN': knn_accuracy,
        'RandomForest': rf_accuracy,
        'SVM': svm_accuracy
    }
    best_model = max(accuracies, key=accuracies.get)
    logging.info(f"Best model is {best_model} with accuracy {accuracies[best_model]}")
    return best_model

## 6. Model Evaluation Component

This component compares the accuracies of the models and selects the best one.

In [11]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn", "numpy"],
    base_image="python:3.10.7-slim"
)
def evaluate_model(
    input_data: Input[Dataset],
    model: Input[Model],
    thresholds_dict_str: str,
    metrics: Output[ClassificationMetrics],
    kpi: Output[Metrics]
) -> NamedTuple('Outputs', [('approval', bool)]):
    """Evaluate the selected model based on accuracy and decide whether to approve it."""
    import pandas as pd
    import pickle
    import numpy as np
    from sklearn.metrics import accuracy_score, confusion_matrix
    import json
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    # Load test data
    data = pd.read_csv(input_data.path + ".csv")
    X = data.drop('quality', axis=1)
    y = data['quality']
    
    # Load the model
    model_file = model.path + model.metadata["file_type"]
    with open(model_file, 'rb') as f:
        loaded_model = pickle.load(f)

    # Make predictions
    y_pred = loaded_model.predict(X)

    # Calculate accuracy
    accuracy = accuracy_score(y, y_pred)
    kpi.log_metric("accuracy", float(accuracy))
    logging.info(f"Accuracy: {accuracy}")

    # Initialize approval as True
    approval = True

    # Load thresholds
    thresholds_dict = json.loads(thresholds_dict_str)
    accuracy_threshold = thresholds_dict.get('accuracy', 0.6)

    if accuracy < accuracy_threshold:
        approval = False
        logging.info(f"Model rejected due to accuracy {accuracy} being lower than threshold {accuracy_threshold}")
    else:
        logging.info(f"Model approved with accuracy {accuracy}")

    # Log confusion matrix
    cm = confusion_matrix(y, y_pred)
    classes = np.unique(y)
    class_names = [str(cls) for cls in classes]
    metrics.log_confusion_matrix(categories=class_names, matrix=cm.tolist())

    from collections import namedtuple
    Outputs = namedtuple('Outputs', ['approval'])
    return Outputs(approval=approval)

## 6. Model Upload Component

In [12]:
@dsl.component(
    packages_to_install=["google-cloud-storage"],
    base_image="python:3.10.7-slim"
)
def upload_model_to_gcs(project_id: str, bucket_name: str, model: Input[Model]):
    """Upload the model to Google Cloud Storage."""
    from google.cloud import storage
    import logging
    import sys

    logging.basicConfig(stream=sys.stdout, level=logging.INFO)

    client = storage.Client(project=project_id)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(f"{model.metadata['model_type']}_model{model.metadata['file_type']}")
    blob.upload_from_filename(model.path + model.metadata['file_type'])
    logging.info(f"Model uploaded to gs://{bucket_name}/{blob.name}")

## 7. Trigger CI-CD Pipeline

In [16]:
@dsl.component(
    packages_to_install=["google-cloud-build"],
    base_image="python:3.10.7-slim"
)
def trigger_cicd_pipeline(project_id:str, trigger_id:str):
    import sys
    from google.cloud.devtools import cloudbuild_v1    
    import logging 
    logging.basicConfig(stream=sys.stdout, level=logging.INFO) 
    
    # Create a client
    client = cloudbuild_v1.CloudBuildClient()
    name = f"projects/{project_id}/locations/us-central1/triggers/{trigger_id}"
    # Initialize request argument(s)
    request = cloudbuild_v1.RunBuildTriggerRequest(        
        project_id=project_id,
        trigger_id=trigger_id,
        name=name
    )

    # Make the request
    operation = client.run_build_trigger(request=request)
    
    logging.info("Trigger the CI-CD Pipeline: " + trigger_id)

## 8. Pipeline Definition

In [17]:
@dsl.pipeline(
    name="red-wine-quality-pipeline"
)
def red_wine_pipeline(
    project_id: str,
    bucket_name: str,
    dataset_file_name: str,
    model_bucket_name: str,
    thresholds_dict_str: str,
    trigger_id: str
):
    # Step 1: Download Data
    data_ingestion = download_data_from_gcs(
        project_id=project_id,
        bucket_name=bucket_name,
        file_name=dataset_file_name
    )

    # Step 2: Split Data
    data_splitting = split_data(
        input_data=data_ingestion.outputs["dataset"]
    )

    # Step 3: Preprocess Data
    data_preprocessing = preprocess_data(
        train_data=data_splitting.outputs["train_data"],
        test_data=data_splitting.outputs["test_data"]
    )

    # Step 4: Train Models
    knn_training = train_knn(
        input_data=data_preprocessing.outputs["processed_train_data"]
    )

    rf_training = train_random_forest(
        input_data=data_preprocessing.outputs["processed_train_data"]
    )

    svm_training = train_svm(
        input_data=data_preprocessing.outputs["processed_train_data"]
    )

    # Step 5: Select Best Model
    model_selection = select_best_model(
        knn_accuracy=knn_training.outputs["accuracy"],
        rf_accuracy=rf_training.outputs["accuracy"],
        svm_accuracy=svm_training.outputs["accuracy"]
    )

    # Step 6: Evaluate Selected Model

    with dsl.If(model_selection.output == "KNN"):
        # Evaluate Model
        evaluation = evaluate_model(
            input_data=data_preprocessing.outputs["processed_test_data"],
            model=knn_training.outputs["knn_model"],
            thresholds_dict_str=thresholds_dict_str
        )

        with dsl.If(evaluation.outputs["approval"] == True):
            # Upload Model
            upload_knn = upload_model_to_gcs(
                project_id=project_id,
                bucket_name=model_bucket_name,
                model=knn_training.outputs["knn_model"]
            )

            # Trigger CI/CD Pipeline
            trigger_cicd = trigger_cicd_pipeline(
                project_id=project_id,
                trigger_id=trigger_id
            ).after(upload_knn)

    with dsl.If(model_selection.output == "RandomForest"):
        # Evaluate Model
        evaluation = evaluate_model(
            input_data=data_preprocessing.outputs["processed_test_data"],
            model=rf_training.outputs["rf_model"],
            thresholds_dict_str=thresholds_dict_str
        )

        with dsl.If(evaluation.outputs["approval"] == True):
            # Upload Model
            upload_rf = upload_model_to_gcs(
                project_id=project_id,
                bucket_name=model_bucket_name,
                model=rf_training.outputs["rf_model"]
            )

            # Trigger CI/CD Pipeline
            trigger_cicd = trigger_cicd_pipeline(
                project_id=project_id,
                trigger_id=trigger_id
            ).after(upload_rf)

    with dsl.If(model_selection.output == "SVM"):
        # Evaluate Model
        evaluation = evaluate_model(
            input_data=data_preprocessing.outputs["processed_test_data"],
            model=svm_training.outputs["svm_model"],
            thresholds_dict_str=thresholds_dict_str
        )

        with dsl.If(evaluation.outputs["approval"] == True):
            # Upload Model
            upload_svm = upload_model_to_gcs(
                project_id=project_id,
                bucket_name=model_bucket_name,
                model=svm_training.outputs["svm_model"]
            )

            # Trigger CI/CD Pipeline
            trigger_cicd = trigger_cicd_pipeline(
                project_id=project_id,
                trigger_id=trigger_id
            ).after(upload_svm)

## 8. Compile the Pipline

In [18]:
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=red_wine_pipeline,
    package_path='red_wine_quality_pipeline.yaml'
)

## 9. Submit Pipline Run

In [ ]:
import google.cloud.aiplatform as aip

# Initialize Vertex AI
aip.init(
    project=PROJECT_ID,
    location=REGION,
)

# Define pipeline parameters
pipeline_parameters = {
    'project_id': PROJECT_ID,
    'bucket_name': 'your-data-bucket',
    'dataset_file_name': 'winequality-red.csv',
    'model_bucket_name': 'your-model-bucket',
    'thresholds_dict_str': '{"accuracy": 0.6}',
    'trigger_id': 'your-cloud-build-trigger-id'  # Replace with your actual trigger ID
}

# Submit the pipeline job
job = aip.PipelineJob(
    display_name="red-wine-quality-pipeline",
    template_path="red_wine_quality_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values=pipeline_parameters,
    enable_caching=False
)

job.run()

Creating PipelineJob
PipelineJob created. Resource name: projects/783782846539/locations/us-central1/pipelineJobs/red-wine-quality-pipeline-20241025175239
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/783782846539/locations/us-central1/pipelineJobs/red-wine-quality-pipeline-20241025175239')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/red-wine-quality-pipeline-20241025175239?project=783782846539
PipelineJob projects/783782846539/locations/us-central1/pipelineJobs/red-wine-quality-pipeline-20241025175239 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/783782846539/locations/us-central1/pipelineJobs/red-wine-quality-pipeline-20241025175239 current state:
PipelineState.PIPELINE_STATE_RUNNING
